# Phase 2 — Feature Engineering
**Input:** `Dataset/Processed/df_identity_clean.csv` (output from Phase 1)  
**Output:** `Dataset/Processed/features.csv`  

**What this notebook does:**
1. Encode categorical string columns into integers
2. Build per-user behavioral profiles (action count, error rate, etc.)
3. Normalize numerical features with MinMaxScaler
4. Merge everything into a final feature matrix
5. Sanity check labels and save output

## Cell 1 — Imports & Load Data

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import os

# update paths to match your local setup
INPUT_PATH  = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv")
OUTPUT_PATH = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features.csv")

df = pd.read_csv(INPUT_PATH)
print("Loaded shape:", df.shape)
df.head(3)

Loaded shape: (2900, 9)


,userName,sourceIPAddress,eventSource,eventName,readOnly,errorCode,type,sessionContext.attributes.mfaAuthenticated,is_anomaly
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,False,NoError,IAMUser,Unknown,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,True,NoError,IAMUser,Unknown,0
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,True,NoError,IAMUser,Unknown,0


## Cell 2 — Confirm Required Columns Exist

In [3]:
REQUIRED_COLS = [
    'userName', 'sourceIPAddress', 'eventSource', 'eventName',
    'type', 'accessKeyId',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'awsRegion', 'eventType', 'readOnly', 'errorCode',
    'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours',
    'is_anomaly'
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    print("WARNING — These columns are missing, check Phase 1 output:")
    for c in missing_cols:
        print("  -", c)
else:
    print("All required columns present. Ready to proceed.")

WARNING — These columns are missing, check Phase 1 output:
  - accessKeyId
  - sessionContext.sessionIssuer.type
  - awsRegion
  - eventType
  - event_hour
  - event_dayofweek
  - is_weekend
  - is_off_hours


## Cell 3 — Encode Categorical Columns
LabelEncoder maps each unique string to an integer.  
Example: `'benjamin' → 0`, `'bert-jan' → 1`  
We save all encoders in a dict so we can reverse-lookup the original labels later.

In [4]:
CATEGORICAL_COLS = [
    'userName',
    'sourceIPAddress',
    'eventSource',
    'eventName',
    'type',
    'sessionContext.attributes.mfaAuthenticated',
    'errorCode'
]

encoders = {}

for col in CATEGORICAL_COLS:
    if col not in df.columns:
        print(f"Skipping '{col}' — not found in dataframe")
        continue
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique classes -> encoded")

# readOnly is already boolean — convert to 0/1 directly
if 'readOnly' in df.columns:
    df['readOnly_enc'] = df['readOnly'].astype(int)
    print("readOnly: bool -> int")

print("\nEncoding complete.")

userName: 4 unique classes -> encoded
sourceIPAddress: 16 unique classes -> encoded
eventSource: 29 unique classes -> encoded
eventName: 260 unique classes -> encoded
type: 4 unique classes -> encoded
sessionContext.attributes.mfaAuthenticated: 3 unique classes -> encoded
errorCode: 33 unique classes -> encoded
readOnly: bool -> int

Encoding complete.


In [5]:
df.head(10)

,userName,sourceIPAddress,eventSource,eventName,readOnly,errorCode,type,sessionContext.attributes.mfaAuthenticated,is_anomaly,userName_enc,sourceIPAddress_enc,eventSource_enc,eventName_enc,type_enc,sessionContext.attributes.mfaAuthenticated_enc,errorCode_enc,readOnly_enc
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,False,NoError,IAMUser,Unknown,0,2,4,9,27,2,0,20,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,True,NoError,IAMUser,Unknown,0,2,4,9,187,2,0,20,1
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,True,NoError,IAMUser,Unknown,0,2,4,9,215,2,0,20,1
3,bert-jan,192.168.10.20,iam.amazonaws.com,ListAttachedRolePolicies,True,NoError,IAMUser,Unknown,0,2,4,9,198,2,0,20,1
4,bert-jan,192.168.10.20,ec2.amazonaws.com,AssociateRouteTable,False,NoError,IAMUser,Unknown,0,2,4,5,3,2,0,20,0
5,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeRouteTables,True,NoError,IAMUser,Unknown,0,2,4,5,122,2,0,20,1
6,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeRouteTables,True,NoError,IAMUser,Unknown,0,2,4,5,122,2,0,20,1
7,bert-jan,192.168.10.20,sts.amazonaws.com,AssumeRole,True,NoError,IAMUser,Unknown,1,2,4,28,4,2,0,20,1
8,bert-jan,192.168.10.20,sts.amazonaws.com,AssumeRole,True,NoError,IAMUser,Unknown,1,2,4,28,4,2,0,20,1
9,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeVpcClassicLinkDnsSupport,True,NoError,IAMUser,Unknown,0,2,4,5,133,2,0,20,1


In [6]:
print(df.shape)

print(df.columns)

(2900, 17)
Index(['userName', 'sourceIPAddress', 'eventSource', 'eventName', 'readOnly',
       'errorCode', 'type', 'sessionContext.attributes.mfaAuthenticated',
       'is_anomaly', 'userName_enc', 'sourceIPAddress_enc', 'eventSource_enc',
       'eventName_enc', 'type_enc',
       'sessionContext.attributes.mfaAuthenticated_enc', 'errorCode_enc',
       'readOnly_enc'],
      dtype='str')


## Cell 4 — Build User Behavior Profile
Instead of looking at each event in isolation, we compute aggregate statistics per user.  
This captures behavioral patterns — e.g. how many services does this user normally access?  
These statistics become the **node feature vectors** for User nodes in the HGNN graph.

In [7]:
user_profile = df.groupby('userName').agg(
    action_count = ('eventName', 'count'),
    unique_ips = ('sourceIPAddress', 'nunique'),
    unique_services = ('eventSource', 'nunique'),
    unique_events = ('eventName', 'nunique'),
    error_count = ('errorCode', lambda x: (x != 'NoError').sum()),
    readonly_ratio = ('readOnly', lambda x: x.astype(int).mean()),
).reset_index()

# Derived rate features — more meaningful than raw counts for comparing users
user_profile['error_rate'] = (user_profile['error_count'] / user_profile['action_count']).round(4)

# Anomaly label per user: flag user as anomalous if ANY of their events was anomalous
user_anomaly = df.groupby('userName')['is_anomaly'].max().reset_index()
user_anomaly.columns = ['userName', 'user_is_anomaly']
user_profile = user_profile.merge(user_anomaly, on='userName', how='left')

print("User behavior profiles:")
user_profile

User behavior profiles:


,userName,action_count,unique_ips,unique_services,unique_events,error_count,readonly_ratio,error_rate,user_is_anomaly
0,Unknown,152,10,6,21,47,0.572368,0.3092,1
1,benjamin,105,4,6,20,14,1.000000,0.1333,0
2,bert-jan,2642,6,27,243,239,0.807721,0.0905,1
3,stratus-red-team-nmfalu-gfjyeaypjt,1,1,1,1,0,0.000000,0.0000,0


## Cell 5 — Normalize Numerical Features
MinMaxScaler squeezes all values into the range [0, 1].  
This prevents `action_count` (e.g. 2642) from dominating `error_rate` (e.g. 0.09).

In [8]:
NUMERICAL_COLS = [
    'action_count',
    'unique_ips',
    'unique_services',
    'unique_events',
    'error_rate',
    'readonly_ratio',
]

scaler = MinMaxScaler()
user_profile[NUMERICAL_COLS] = scaler.fit_transform(user_profile[NUMERICAL_COLS])

print("After normalization (all values should be between 0.0 and 1.0):")
user_profile[NUMERICAL_COLS].describe().round(4)

After normalization (all values should be between 0.0 and 1.0):


,action_count,unique_ips,unique_services,unique_events,error_rate,readonly_ratio
count,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000
mean,0.2741,0.4722,0.3462,0.2903,0.4310,0.5950
std,0.4845,0.4194,0.4452,0.4747,0.4198,0.4335
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0295,0.2500,0.1442,0.0589,0.2195,0.4293
50%,0.0483,0.4444,0.1923,0.0806,0.3619,0.6900
75%,0.2929,0.6667,0.3942,0.3120,0.5733,0.8558
max,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


## Cell 6 — Merge Profile Back + Build Final Feature Matrix
Each event row now also carries the behavioral profile of the user who triggered it.  
This is the final feature matrix — one row per event, all values as numbers.

In [9]:
df = df.merge(
    user_profile[['userName'] + NUMERICAL_COLS + ['user_is_anomaly']],
    on='userName',
    how='left',
    suffixes=('', '_profile')
)

FEATURE_COLS = [
    # Encoded identity (categorical -> numerical)
    'userName_enc',
    'sourceIPAddress_enc',
    'eventSource_enc',
    'eventName_enc',
    'type_enc',
    'sessionContext.attributes.mfaAuthenticated_enc',
    'errorCode_enc',
    'readOnly_enc',
    
    # User behavior profile (normalized stats)
    'action_count',
    'unique_ips',
    'unique_services',
    'unique_events',
    'error_rate',
    'readonly_ratio',
    
    # Target Label
    'is_anomaly'
]

# Only keep columns that actually exist (safety check)
final_cols = [c for c in FEATURE_COLS if c in df.columns]
df_features = df[final_cols].copy()

print("Final feature matrix shape:", df_features.shape)
print("\nMissing values in final matrix:")
missing = df_features.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — all clean.")
print("\nSample:")
df_features.head(5)

Final feature matrix shape: (2900, 15)

Missing values in final matrix:
None — all clean.

Sample:


,userName_enc,sourceIPAddress_enc,eventSource_enc,eventName_enc,type_enc,sessionContext.attributes.mfaAuthenticated_enc,errorCode_enc,readOnly_enc,action_count,unique_ips,unique_services,unique_events,error_rate,readonly_ratio,is_anomaly
0,2,4,9,27,2,0,20,0,1.0,0.555556,1.0,1.0,0.292691,0.807721,0
1,2,4,9,187,2,0,20,1,1.0,0.555556,1.0,1.0,0.292691,0.807721,0
2,2,4,9,215,2,0,20,1,1.0,0.555556,1.0,1.0,0.292691,0.807721,0
3,2,4,9,198,2,0,20,1,1.0,0.555556,1.0,1.0,0.292691,0.807721,0
4,2,4,5,3,2,0,20,0,1.0,0.555556,1.0,1.0,0.292691,0.807721,0


In [10]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 15 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   userName_enc                                    2900 non-null   int64  
 1   sourceIPAddress_enc                             2900 non-null   int64  
 2   eventSource_enc                                 2900 non-null   int64  
 3   eventName_enc                                   2900 non-null   int64  
 4   type_enc                                        2900 non-null   int64  
 5   sessionContext.attributes.mfaAuthenticated_enc  2900 non-null   int64  
 6   errorCode_enc                                   2900 non-null   int64  
 7   readOnly_enc                                    2900 non-null   int64  
 8   action_count                                    2900 non-null   float64
 9   unique_ips                                      2900

In [11]:
df_features.describe()

,userName_enc,sourceIPAddress_enc,eventSource_enc,eventName_enc,type_enc,sessionContext.attributes.mfaAuthenticated_enc,errorCode_enc,readOnly_enc,action_count,unique_ips,unique_services,unique_events,error_rate,readonly_ratio,is_anomaly
count,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000,2900.000000
mean,1.859310,4.622414,14.059655,130.879655,1.964828,0.355862,20.171034,0.802069,0.915457,0.570613,0.928077,0.918209,0.334674,0.802069,0.041379
std,0.475911,2.525041,8.860552,63.372743,0.293903,0.690134,3.289062,0.398509,0.270602,0.109659,0.230225,0.261786,0.158718,0.066606,0.199200
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,4.000000,5.000000,86.000000,2.000000,0.000000,20.000000,1.000000,1.000000,0.555556,1.000000,1.000000,0.292691,0.807721,0.000000
50%,2.000000,4.000000,10.000000,124.000000,2.000000,0.000000,20.000000,1.000000,1.000000,0.555556,1.000000,1.000000,0.292691,0.807721,0.000000
75%,2.000000,4.000000,23.000000,186.000000,2.000000,0.000000,20.000000,1.000000,1.000000,0.555556,1.000000,1.000000,0.292691,0.807721,0.000000
max,3.000000,15.000000,28.000000,259.000000,3.000000,2.000000,32.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Cell 7 — Sanity Check: Label Distribution
Confirm the anomaly label is non-trivial.  
Healthy range for a prototype: **5–30% anomaly rate**.  
If result is 0%, go back to Phase 1 Cell 6 and check `HIGH_RISK_EVENTS`.

In [12]:
label_counts = df_features['is_anomaly'].value_counts()
total = len(df_features)

print("Label distribution:")
print(f"  Normal   (0): {label_counts.get(0, 0):>5} rows  ({label_counts.get(0,0)/total*100:.1f}%)")
print(f"  Anomaly  (1): {label_counts.get(1, 0):>5} rows  ({label_counts.get(1,0)/total*100:.1f}%)")

print("\nAnomaly events breakdown (what triggered the flag):")
anomaly_events = df[df['is_anomaly'] == 1]['eventName'].value_counts()
print(anomaly_events.to_string())

print("\nAnomaly rate per user:")
print(df.groupby('userName')['is_anomaly'].mean().round(4).to_string())

Label distribution:
  Normal   (0):  2780 rows  (95.9%)
  Anomaly  (1):   120 rows  (4.1%)

Anomaly events breakdown (what triggered the flag):
eventName
GetSecretValue        60
AssumeRole            49
AttachRolePolicy       6
CreateLoginProfile     2
CreateAccessKey        2
AttachUserPolicy       1

Anomaly rate per user:
userName
Unknown                               0.1711
benjamin                              0.0000
bert-jan                              0.0356
stratus-red-team-nmfalu-gfjyeaypjt    0.0000


## Cell 8 — Save Output

In [13]:
# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df_features.shape}")
print(f"Columns: {df_features.columns.tolist()}")

Saved: /Users/philberttan/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features.csv
Shape: (2900, 15)
Columns: ['userName_enc', 'sourceIPAddress_enc', 'eventSource_enc', 'eventName_enc', 'type_enc', 'sessionContext.attributes.mfaAuthenticated_enc', 'errorCode_enc', 'readOnly_enc', 'action_count', 'unique_ips', 'unique_services', 'unique_events', 'error_rate', 'readonly_ratio', 'is_anomaly']
